In [2]:
# Load env variables

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [20]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Make a request

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages":  messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    
    return message.content[0].text

In [ ]:
# Multi-Turn conversations

# Make a starting list of messages
messages = []

# Add in the initial user question of "Define quantum computing in one sentence"
add_user_message(messages, "Define quantum computing in one sentence")

# Pass the list of messages into 'chat' to get an answer
answer = chat(messages)

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

messages
# Add in the user's follow-up question
add_user_message(messages, "Write another sentence")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)

answer

'Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits which are limited to being either 0 or 1.'

In [ ]:
# Chat exercise

# Make an initial list of messages
messages = []

# Use a 'While True' loop to run the chatbot forever
while True:
    # Get user input
    user_input = input ("> ")
    print(">", user_input)

    # Add user input to the list of messages
    add_user_message(messages, user_input)

    # Call Claude with the 'chat' function
    answer = chat(messages)

    # Add generated text to the list of messages
    add_assistant_message(messages, answer)

    # Print the generated text
    print("----")
    print(answer)
    print("----")


> what is 1+1
----
1 + 1 = 2
----
> add 1 more
----
2 + 1 = 3
----


In [ ]:
# System prompts

messages = []
system = """
    You are a patient math tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step
    """

add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages, system=system)

answer

'I\'d be happy to help you solve this equation step by step! \n\nLet\'s start by looking at what we have: 5x + 3 = 2\n\nThe goal is to get x by itself on one side of the equation. To do this, we need to "undo" the operations that are being done to x.\n\nLooking at the left side, what two things are happening to x? And in what order?\n\nOnce you identify those operations, we can start undoing them one at a time. What do you think should be our first step?'

In [14]:
# Temperature

messages = []

add_user_message(messages, "Generate a one sentence tamil movie idea")
answer = chat(messages, temperature = 1.0)

answer

"A young street food vendor discovers that his grandmother's secret spice blend can make people temporarily forget their deepest sorrows, leading him on a journey to heal his war-torn village while learning that some pain must be faced to find true peace."

In [ ]:
# stream = true
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    stream = True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01EPoi8Wv6XgxqsyZMSKzLEG', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard'), stop_details=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeShop Pro is a synthetic e-commerce database containing 50,000 randomly', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' generated custom

In [17]:
# stream - get text
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model = model,
    max_tokens = 1000,
    messages = messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

stream.get_final_message()

The "GlobalSync Personal Data Warehouse" is a fictional database that claims to store comprehensive behavioral profiles, purchase histories, and real-time location data for over 2 billion users across 50 countries, marketed as an all-in-one consumer insights platform.

ParsedMessage(id='msg_01Qubd2ZL7aoM5LNf9oE5c5d', container=None, content=[ParsedTextBlock(citations=None, text='The "GlobalSync Personal Data Warehouse" is a fictional database that claims to store comprehensive behavioral profiles, purchase histories, and real-time location data for over 2 billion users across 50 countries, marketed as an all-in-one consumer insights platform.', type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=58, server_tool_use=None, service_tier='standard'), stop_details=None)

In [21]:
# structured data - getting just the data

import json

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

json.loads(text.strip())

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'State': 'ENABLED',
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

In [29]:
# structured data - exercise

from IPython.display import Markdown

messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

text = chat(messages, stop_sequences=["```"])
text.strip()

#Markdown(text.strip())

'aws s3 ls\n\naws ec2 describe-instances\n\naws iam list-users'